## 1. Introduction
This notebook presents the full workflow for the HR Analytics clustering project, including data preparation, exploratory analysis, clustering models, evaluation, and final insights.

## 2. Import Libraries

In [100]:
import pandas as pd
from pathlib import Path

## 3. Load and Merge Data

In this section, we inspect the raw HR datasets, standardize column names, identify the common merge key, and combine the main employee-related tables into one merged dataset.

In [101]:
# Set the path to the raw data folder

DATA_DIR = Path("../data/raw")

print(DATA_DIR)
print(DATA_DIR.exists())

..\data\raw
True


In [102]:
# List all CSV files in the raw data directory

csv_files = list(DATA_DIR.glob("*.csv"))

print("Number of CSV files:", len(csv_files))
for file in csv_files:
    print(file.name)

Number of CSV files: 5
employee_survey_data.csv
general_data.csv
in_time.csv
manager_survey_data.csv
out_time.csv


In [103]:
# Inspect each raw CSV file: file name, shape, columns, and first 5 rows

for file in csv_files:
    print("=" * 70)
    print("FILE NAME:", file.name)

    df = pd.read_csv(file)

    print("SHAPE:", df.shape)
    print("COLUMNS:")
    print(list(df.columns))

    print("\nFIRST 5 ROWS:")
    display(df.head())

FILE NAME: employee_survey_data.csv
SHAPE: (4410, 4)
COLUMNS:
['EmployeeID', 'EnvironmentSatisfaction', 'JobSatisfaction', 'WorkLifeBalance']

FIRST 5 ROWS:


,EmployeeID,EnvironmentSatisfaction,JobSatisfaction,WorkLifeBalance
0,1,3.0,4.0,2.0
1,2,3.0,2.0,4.0
2,3,2.0,2.0,1.0
3,4,4.0,4.0,3.0
4,5,4.0,1.0,3.0


FILE NAME: general_data.csv
SHAPE: (4410, 24)
COLUMNS:
['Age', 'Attrition', 'BusinessTravel', 'Department', 'DistanceFromHome', 'Education', 'EducationField', 'EmployeeCount', 'EmployeeID', 'Gender', 'JobLevel', 'JobRole', 'MaritalStatus', 'MonthlyIncome', 'NumCompaniesWorked', 'Over18', 'PercentSalaryHike', 'StandardHours', 'StockOptionLevel', 'TotalWorkingYears', 'TrainingTimesLastYear', 'YearsAtCompany', 'YearsSinceLastPromotion', 'YearsWithCurrManager']

FIRST 5 ROWS:


,Age,Attrition,BusinessTravel,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeID,Gender,...,NumCompaniesWorked,Over18,PercentSalaryHike,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,YearsAtCompany,YearsSinceLastPromotion,YearsWithCurrManager
0,51,No,Travel_Rarely,Sales,6,2,Life Sciences,1,1,Female,...,1.0,Y,11,8,0,1.0,6,1,0,0
1,31,Yes,Travel_Frequently,Research & Development,10,1,Life Sciences,1,2,Female,...,0.0,Y,23,8,1,6.0,3,5,1,4
2,32,No,Travel_Frequently,Research & Development,17,4,Other,1,3,Male,...,1.0,Y,15,8,3,5.0,2,5,0,3
3,38,No,Non-Travel,Research & Development,2,5,Life Sciences,1,4,Male,...,3.0,Y,11,8,3,13.0,5,8,7,5
4,32,No,Travel_Rarely,Research & Development,10,1,Medical,1,5,Male,...,4.0,Y,12,8,2,9.0,2,6,0,4


FILE NAME: in_time.csv
SHAPE: (4410, 262)
COLUMNS:
['Unnamed: 0', '2015-01-01', '2015-01-02', '2015-01-05', '2015-01-06', '2015-01-07', '2015-01-08', '2015-01-09', '2015-01-12', '2015-01-13', '2015-01-14', '2015-01-15', '2015-01-16', '2015-01-19', '2015-01-20', '2015-01-21', '2015-01-22', '2015-01-23', '2015-01-26', '2015-01-27', '2015-01-28', '2015-01-29', '2015-01-30', '2015-02-02', '2015-02-03', '2015-02-04', '2015-02-05', '2015-02-06', '2015-02-09', '2015-02-10', '2015-02-11', '2015-02-12', '2015-02-13', '2015-02-16', '2015-02-17', '2015-02-18', '2015-02-19', '2015-02-20', '2015-02-23', '2015-02-24', '2015-02-25', '2015-02-26', '2015-02-27', '2015-03-02', '2015-03-03', '2015-03-04', '2015-03-05', '2015-03-06', '2015-03-09', '2015-03-10', '2015-03-11', '2015-03-12', '2015-03-13', '2015-03-16', '2015-03-17', '2015-03-18', '2015-03-19', '2015-03-20', '2015-03-23', '2015-03-24', '2015-03-25', '2015-03-26', '2015-03-27', '2015-03-30', '2015-03-31', '2015-04-01', '2015-04-02', '2015-04-0

,Unnamed: 0,2015-01-01,2015-01-02,2015-01-05,2015-01-06,2015-01-07,2015-01-08,2015-01-09,2015-01-12,2015-01-13,...,2015-12-18,2015-12-21,2015-12-22,2015-12-23,2015-12-24,2015-12-25,2015-12-28,2015-12-29,2015-12-30,2015-12-31
0,1,NaN,2015-01-02 09:43:45,2015-01-05 10:08:48,2015-01-06 09:54:26,2015-01-07 09:34:31,2015-01-08 09:51:09,2015-01-09 10:09:25,2015-01-12 09:42:53,2015-01-13 10:13:06,...,NaN,2015-12-21 09:55:29,2015-12-22 10:04:06,2015-12-23 10:14:27,2015-12-24 10:11:35,NaN,2015-12-28 10:13:41,2015-12-29 10:03:36,2015-12-30 09:54:12,2015-12-31 10:12:44
1,2,NaN,2015-01-02 10:15:44,2015-01-05 10:21:05,NaN,2015-01-07 09:45:17,2015-01-08 10:09:04,2015-01-09 09:43:26,2015-01-12 10:00:07,2015-01-13 10:43:29,...,2015-12-18 10:37:17,2015-12-21 09:49:02,2015-12-22 10:33:51,2015-12-23 10:12:10,NaN,NaN,2015-12-28 09:31:45,2015-12-29 09:55:49,2015-12-30 10:32:25,2015-12-31 09:27:20
2,3,NaN,2015-01-02 10:17:41,2015-01-05 09:50:50,2015-01-06 10:14:13,2015-01-07 09:47:27,2015-01-08 10:03:40,2015-01-09 10:05:49,2015-01-12 10:03:47,2015-01-13 10:21:26,...,2015-12-18 10:15:14,2015-12-21 10:10:28,2015-12-22 09:44:44,2015-12-23 10:15:54,2015-12-24 10:07:26,NaN,2015-12-28 09:42:05,2015-12-29 09:43:36,2015-12-30 09:34:05,2015-12-31 10:28:39
3,4,NaN,2015-01-02 10:05:06,2015-01-05 09:56:32,2015-01-06 10:11:07,2015-01-07 09:37:30,2015-01-08 10:02:08,2015-01-09 10:08:12,2015-01-12 10:13:42,2015-01-13 09:53:22,...,2015-12-18 10:17:38,2015-12-21 09:58:21,2015-12-22 10:04:25,2015-12-23 10:11:46,2015-12-24 09:43:15,NaN,2015-12-28 09:52:44,2015-12-29 09:33:16,2015-12-30 10:18:12,2015-12-31 10:01:15
4,5,NaN,2015-01-02 10:28:17,2015-01-05 09:49:58,2015-01-06 09:45:28,2015-01-07 09:49:37,2015-01-08 10:19:44,2015-01-09 10:00:50,2015-01-12 10:29:27,2015-01-13 09:59:32,...,2015-12-18 09:58:35,2015-12-21 10:03:41,2015-12-22 10:10:30,2015-12-23 10:13:36,2015-12-24 09:44:24,NaN,2015-12-28 10:05:15,2015-12-29 10:30:53,2015-12-30 09:18:21,2015-12-31 09:41:09


FILE NAME: manager_survey_data.csv
SHAPE: (4410, 3)
COLUMNS:
['EmployeeID', 'JobInvolvement', 'PerformanceRating']

FIRST 5 ROWS:


,EmployeeID,JobInvolvement,PerformanceRating
0,1,3,3
1,2,2,4
2,3,3,3
3,4,2,3
4,5,3,3


FILE NAME: out_time.csv
SHAPE: (4410, 262)
COLUMNS:
['Unnamed: 0', '2015-01-01', '2015-01-02', '2015-01-05', '2015-01-06', '2015-01-07', '2015-01-08', '2015-01-09', '2015-01-12', '2015-01-13', '2015-01-14', '2015-01-15', '2015-01-16', '2015-01-19', '2015-01-20', '2015-01-21', '2015-01-22', '2015-01-23', '2015-01-26', '2015-01-27', '2015-01-28', '2015-01-29', '2015-01-30', '2015-02-02', '2015-02-03', '2015-02-04', '2015-02-05', '2015-02-06', '2015-02-09', '2015-02-10', '2015-02-11', '2015-02-12', '2015-02-13', '2015-02-16', '2015-02-17', '2015-02-18', '2015-02-19', '2015-02-20', '2015-02-23', '2015-02-24', '2015-02-25', '2015-02-26', '2015-02-27', '2015-03-02', '2015-03-03', '2015-03-04', '2015-03-05', '2015-03-06', '2015-03-09', '2015-03-10', '2015-03-11', '2015-03-12', '2015-03-13', '2015-03-16', '2015-03-17', '2015-03-18', '2015-03-19', '2015-03-20', '2015-03-23', '2015-03-24', '2015-03-25', '2015-03-26', '2015-03-27', '2015-03-30', '2015-03-31', '2015-04-01', '2015-04-02', '2015-04-

,Unnamed: 0,2015-01-01,2015-01-02,2015-01-05,2015-01-06,2015-01-07,2015-01-08,2015-01-09,2015-01-12,2015-01-13,...,2015-12-18,2015-12-21,2015-12-22,2015-12-23,2015-12-24,2015-12-25,2015-12-28,2015-12-29,2015-12-30,2015-12-31
0,1,NaN,2015-01-02 16:56:15,2015-01-05 17:20:11,2015-01-06 17:19:05,2015-01-07 16:34:55,2015-01-08 17:08:32,2015-01-09 17:38:29,2015-01-12 16:58:39,2015-01-13 18:02:58,...,NaN,2015-12-21 17:15:50,2015-12-22 17:27:51,2015-12-23 16:44:44,2015-12-24 17:47:22,NaN,2015-12-28 18:00:07,2015-12-29 17:22:30,2015-12-30 17:40:56,2015-12-31 17:17:33
1,2,NaN,2015-01-02 18:22:17,2015-01-05 17:48:22,NaN,2015-01-07 17:09:06,2015-01-08 17:34:04,2015-01-09 16:52:29,2015-01-12 17:36:48,2015-01-13 18:00:13,...,2015-12-18 18:31:28,2015-12-21 17:34:16,2015-12-22 18:16:35,2015-12-23 17:38:18,NaN,NaN,2015-12-28 17:08:38,2015-12-29 17:54:46,2015-12-30 18:31:35,2015-12-31 17:40:58
2,3,NaN,2015-01-02 16:59:14,2015-01-05 17:06:46,2015-01-06 16:38:32,2015-01-07 16:33:21,2015-01-08 17:24:22,2015-01-09 16:57:30,2015-01-12 17:28:54,2015-01-13 17:21:25,...,2015-12-18 17:02:23,2015-12-21 17:20:17,2015-12-22 16:32:50,2015-12-23 16:59:43,2015-12-24 16:58:25,NaN,2015-12-28 16:43:31,2015-12-29 17:09:56,2015-12-30 17:06:25,2015-12-31 17:15:50
3,4,NaN,2015-01-02 17:25:24,2015-01-05 17:14:03,2015-01-06 17:07:42,2015-01-07 16:32:40,2015-01-08 16:53:11,2015-01-09 17:19:47,2015-01-12 17:13:37,2015-01-13 17:11:45,...,2015-12-18 17:55:23,2015-12-21 16:49:09,2015-12-22 17:24:00,2015-12-23 17:36:35,2015-12-24 16:48:21,NaN,2015-12-28 17:19:34,2015-12-29 16:58:16,2015-12-30 17:40:11,2015-12-31 17:09:14
4,5,NaN,2015-01-02 18:31:37,2015-01-05 17:49:15,2015-01-06 17:26:25,2015-01-07 17:37:59,2015-01-08 17:59:28,2015-01-09 17:44:08,2015-01-12 18:51:21,2015-01-13 18:14:58,...,2015-12-18 17:52:48,2015-12-21 17:43:35,2015-12-22 18:07:57,2015-12-23 18:00:49,2015-12-24 17:59:22,NaN,2015-12-28 17:44:59,2015-12-29 18:47:00,2015-12-30 17:15:33,2015-12-31 17:42:14


### Initial file inspection notes

Based on the raw data inspection:

- `general_data.csv` appears to be the main employee table because it contains the most complete employee-level information.
- `employee_survey_data.csv` contains employee satisfaction-related variables.
- `manager_survey_data.csv` contains manager evaluation variables.
- `in_time.csv` and `out_time.csv` contain daily attendance timestamps in a wide format.

In [104]:
# Define a helper function to standardize column names

def clean_columns(df):
    df = df.copy()
    df.columns = (
        df.columns
        .str.strip()
        .str.replace(r'(?<=[a-z0-9])(?=[A-Z])', '_', regex=True)
        .str.replace(r'(?<=[A-Z])(?=[A-Z][a-z])', '_', regex=True)
        .str.replace(r'(?<=[A-Za-z])(?=[0-9])', '_', regex=True)
        .str.replace(r'[^A-Za-z0-9]+', '_', regex=True)
        .str.lower()
        .str.strip('_')
    )
    return df

In [105]:
# Load all datasets and standardize their column names

general = clean_columns(pd.read_csv(DATA_DIR / "general_data.csv"))
employee_survey = clean_columns(pd.read_csv(DATA_DIR / "employee_survey_data.csv"))
manager_survey = clean_columns(pd.read_csv(DATA_DIR / "manager_survey_data.csv"))
in_time = clean_columns(pd.read_csv(DATA_DIR / "in_time.csv"))
out_time = clean_columns(pd.read_csv(DATA_DIR / "out_time.csv"))

# Rename the first attendance column to employee_id

in_time = in_time.rename(columns={"unnamed_0": "employee_id"})
out_time = out_time.rename(columns={"unnamed_0": "employee_id"})

In [106]:
# Preview cleaned column names

print("general columns:")
print(general.columns.tolist())

print("\nemployee_survey columns:")
print(employee_survey.columns.tolist())

print("\nmanager_survey columns:")
print(manager_survey.columns.tolist())

print("\nin_time first 5 columns:")
print(in_time.columns.tolist()[:5])

print("\nout_time first 5 columns:")
print(out_time.columns.tolist()[:5])

general columns:
['age', 'attrition', 'business_travel', 'department', 'distance_from_home', 'education', 'education_field', 'employee_count', 'employee_id', 'gender', 'job_level', 'job_role', 'marital_status', 'monthly_income', 'num_companies_worked', 'over_18', 'percent_salary_hike', 'standard_hours', 'stock_option_level', 'total_working_years', 'training_times_last_year', 'years_at_company', 'years_since_last_promotion', 'years_with_curr_manager']

employee_survey columns:
['employee_id', 'environment_satisfaction', 'job_satisfaction', 'work_life_balance']

manager_survey columns:
['employee_id', 'job_involvement', 'performance_rating']

in_time first 5 columns:
['employee_id', '2015_01_01', '2015_01_02', '2015_01_05', '2015_01_06']

out_time first 5 columns:
['employee_id', '2015_01_01', '2015_01_02', '2015_01_05', '2015_01_06']


In [107]:
# Check whether employee_id exists in all relevant tables

print("employee_id in general:", "employee_id" in general.columns)
print("employee_id in employee_survey:", "employee_id" in employee_survey.columns)
print("employee_id in manager_survey:", "employee_id" in manager_survey.columns)
print("employee_id in in_time:", "employee_id" in in_time.columns)
print("employee_id in out_time:", "employee_id" in out_time.columns)

employee_id in general: True
employee_id in employee_survey: True
employee_id in manager_survey: True
employee_id in in_time: True
employee_id in out_time: True


In [108]:
# Check whether employee_id is unique in each table before merging

print("Duplicate employee_id in general:", general["employee_id"].duplicated().sum())
print("Duplicate employee_id in employee_survey:", employee_survey["employee_id"].duplicated().sum())
print("Duplicate employee_id in manager_survey:", manager_survey["employee_id"].duplicated().sum())
print("Duplicate employee_id in in_time:", in_time["employee_id"].duplicated().sum())
print("Duplicate employee_id in out_time:", out_time["employee_id"].duplicated().sum())

Duplicate employee_id in general: 0
Duplicate employee_id in employee_survey: 0
Duplicate employee_id in manager_survey: 0
Duplicate employee_id in in_time: 0
Duplicate employee_id in out_time: 0


In [109]:
# Compare the first few employee IDs across tables

print("general employee_id:", general["employee_id"].head().tolist())
print("employee_survey employee_id:", employee_survey["employee_id"].head().tolist())
print("manager_survey employee_id:", manager_survey["employee_id"].head().tolist())
print("in_time employee_id:", in_time["employee_id"].head().tolist())
print("out_time employee_id:", out_time["employee_id"].head().tolist())

general employee_id: [1, 2, 3, 4, 5]
employee_survey employee_id: [1, 2, 3, 4, 5]
manager_survey employee_id: [1, 2, 3, 4, 5]
in_time employee_id: [1, 2, 3, 4, 5]
out_time employee_id: [1, 2, 3, 4, 5]


### Merge key summary

The common key used for merging the main HR tables is `employee_id`.

- `general_data.csv` ↔ `employee_survey_data.csv`: `employee_id`
- `general_data.csv` ↔ `manager_survey_data.csv`: `employee_id`

The attendance tables also appear to use employee IDs after renaming `unnamed_0` to `employee_id`. However, these two tables are kept separate at this stage because they contain daily timestamps in a very wide format.

In [110]:
# Start with the general employee table as the main table

print("Main table shape:", general.shape)

Main table shape: (4410, 24)


In [111]:
# Merge employee survey data into the main table using a left join
# A left join is used to preserve all employees from the main table

print("Shape before merge 1:", general.shape)

merged_df = general.merge(employee_survey, on="employee_id", how="left")

print("Shape after merge 1:", merged_df.shape)
print("Duplicate employee_id after merge 1:", merged_df["employee_id"].duplicated().sum())

Shape before merge 1: (4410, 24)
Shape after merge 1: (4410, 27)
Duplicate employee_id after merge 1: 0


In [112]:
# Check missing values in the columns added from employee survey data

employee_survey_cols = [
    "environment_satisfaction",
    "job_satisfaction",
    "work_life_balance"
]

print("Missing values after merge 1:")
print(merged_df[employee_survey_cols].isna().sum())

Missing values after merge 1:
environment_satisfaction    25
job_satisfaction            20
work_life_balance           38
dtype: int64


The missing values in the employee survey columns are inherited from the original survey data rather than being caused by the merge process. These missing values will be handled in the Data Cleaning section.

In [113]:
# Merge manager survey data into the intermediate merged table

print("Shape before merge 2:", merged_df.shape)

merged_df = merged_df.merge(manager_survey, on="employee_id", how="left")

print("Shape after merge 2:", merged_df.shape)
print("Duplicate employee_id after merge 2:", merged_df["employee_id"].duplicated().sum())

Shape before merge 2: (4410, 27)
Shape after merge 2: (4410, 29)
Duplicate employee_id after merge 2: 0


In [114]:
# Check missing values in the columns added from manager survey data

manager_survey_cols = [
    "job_involvement",
    "performance_rating"
]

print("Missing values after merge 2:")
print(merged_df[manager_survey_cols].isna().sum())

Missing values after merge 2:
job_involvement       0
performance_rating    0
dtype: int64


In [115]:
# Preview the final merged dataset

display(merged_df.head())
print("Final merged shape:", merged_df.shape)

,age,attrition,business_travel,department,distance_from_home,education,education_field,employee_count,employee_id,gender,...,total_working_years,training_times_last_year,years_at_company,years_since_last_promotion,years_with_curr_manager,environment_satisfaction,job_satisfaction,work_life_balance,job_involvement,performance_rating
0,51,No,Travel_Rarely,Sales,6,2,Life Sciences,1,1,Female,...,1.0,6,1,0,0,3.0,4.0,2.0,3,3
1,31,Yes,Travel_Frequently,Research & Development,10,1,Life Sciences,1,2,Female,...,6.0,3,5,1,4,3.0,2.0,4.0,2,4
2,32,No,Travel_Frequently,Research & Development,17,4,Other,1,3,Male,...,5.0,2,5,0,3,2.0,2.0,1.0,3,3
3,38,No,Non-Travel,Research & Development,2,5,Life Sciences,1,4,Male,...,13.0,5,8,7,5,4.0,4.0,3.0,2,3
4,32,No,Travel_Rarely,Research & Development,10,1,Medical,1,5,Male,...,9.0,2,6,0,4,4.0,1.0,3.0,3,3


Final merged shape: (4410, 29)


After both merges, the number of rows should remain unchanged. This indicates that `employee_id` is a consistent merge key across the main HR tables and that the merge did not create unexpected duplicated employee records.

### Attendance tables

The tables `in_time.csv` and `out_time.csv` are not merged directly into the main dataset at this step.

Reason:
- They are very wide tables with daily timestamp columns.
- They are more suitable for separate preprocessing and feature extraction later, such as:
  - average check-in time
  - average check-out time
  - number of late arrivals
  - number of missing attendance records

In [116]:
# Save the merged raw dataset for the next data cleaning step

PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

merged_df.to_csv(PROCESSED_DIR / "merged_raw.csv", index=False)
print("Saved merged_raw.csv successfully")

Saved merged_raw.csv successfully


## 4. Data Cleaning

In this section, the merged HR dataset is cleaned before exploratory analysis and clustering.  
The cleaning process includes:

- checking missing values and planning imputation
- checking duplicate rows and duplicate employee IDs
- reviewing data types
- cleaning categorical values
- removing uninformative columns
- performing a preliminary outlier check
- saving the cleaned dataset for the next project stages

### 4.1 Missing Values

We first inspect missing values in the merged dataset to understand their extent and decide an appropriate handling strategy before applying any imputation.

In [117]:
# Create a copy of the merged dataset for cleaning

df_clean = merged_df.copy()

print("Current shape:", df_clean.shape)
display(df_clean.head())

Current shape: (4410, 29)


,age,attrition,business_travel,department,distance_from_home,education,education_field,employee_count,employee_id,gender,...,total_working_years,training_times_last_year,years_at_company,years_since_last_promotion,years_with_curr_manager,environment_satisfaction,job_satisfaction,work_life_balance,job_involvement,performance_rating
0,51,No,Travel_Rarely,Sales,6,2,Life Sciences,1,1,Female,...,1.0,6,1,0,0,3.0,4.0,2.0,3,3
1,31,Yes,Travel_Frequently,Research & Development,10,1,Life Sciences,1,2,Female,...,6.0,3,5,1,4,3.0,2.0,4.0,2,4
2,32,No,Travel_Frequently,Research & Development,17,4,Other,1,3,Male,...,5.0,2,5,0,3,2.0,2.0,1.0,3,3
3,38,No,Non-Travel,Research & Development,2,5,Life Sciences,1,4,Male,...,13.0,5,8,7,5,4.0,4.0,3.0,2,3
4,32,No,Travel_Rarely,Research & Development,10,1,Medical,1,5,Male,...,9.0,2,6,0,4,4.0,1.0,3.0,3,3


In [118]:
# Count missing values and calculate missing percentages

missing_count = df_clean.isna().sum()
missing_pct = (df_clean.isna().mean() * 100).round(2)

missing_report = pd.DataFrame({
    "column": df_clean.columns,
    "missing_count": missing_count.values,
    "missing_pct": missing_pct.values
})

# Keep only columns with missing values

missing_report = missing_report[missing_report["missing_count"] > 0]
missing_report = missing_report.sort_values(by="missing_pct", ascending=False).reset_index(drop=True)

display(missing_report)

,column,missing_count,missing_pct
0,work_life_balance,38,0.86
1,environment_satisfaction,25,0.57
2,job_satisfaction,20,0.45
3,num_companies_worked,19,0.43
4,total_working_years,9,0.20


In [119]:
# Identify numeric and categorical columns

id_cols = ["employee_id"]

numeric_cols = df_clean.select_dtypes(include="number").columns.difference(id_cols)
categorical_cols = df_clean.select_dtypes(exclude="number").columns.difference(id_cols)

print("Numeric columns:")
print(list(numeric_cols))

print("\nCategorical columns:")
print(list(categorical_cols))

Numeric columns:
['age', 'distance_from_home', 'education', 'employee_count', 'environment_satisfaction', 'job_involvement', 'job_level', 'job_satisfaction', 'monthly_income', 'num_companies_worked', 'percent_salary_hike', 'performance_rating', 'standard_hours', 'stock_option_level', 'total_working_years', 'training_times_last_year', 'work_life_balance', 'years_at_company', 'years_since_last_promotion', 'years_with_curr_manager']

Categorical columns:
['attrition', 'business_travel', 'department', 'education_field', 'gender', 'job_role', 'marital_status', 'over_18']


In [120]:
# Suggest a simple handling strategy for each column with missing values

def suggest_action(col_name, missing_pct):
    if col_name in numeric_cols:
        if missing_pct > 40:
            return "Discuss drop or special treatment"
        return "Fill with median"
    elif col_name in categorical_cols:
        if missing_pct > 40:
            return "Discuss drop or special treatment"
        return "Fill with mode or 'Unknown'"
    else:
        return "Check manually"

missing_report["suggested_action"] = missing_report.apply(
    lambda row: suggest_action(row["column"], row["missing_pct"]),
    axis=1
)

display(missing_report)

,column,missing_count,missing_pct,suggested_action
0,work_life_balance,38,0.86,Fill with median
1,environment_satisfaction,25,0.57,Fill with median
2,job_satisfaction,20,0.45,Fill with median
3,num_companies_worked,19,0.43,Fill with median
4,total_working_years,9,0.20,Fill with median


In [121]:
# Apply missing value imputation based on the cleaning plan

for col in missing_report["column"]:
    if col in numeric_cols:
        df_clean[col] = df_clean[col].fillna(df_clean[col].median())
    elif col in categorical_cols:
        df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])

The missing values are limited to only a few columns and all missing rates are below 1%.  
Since the affected columns are numeric, median imputation is used to reduce the influence of extreme values.

### 4.2 Duplicate Check

We check duplicates both at the full-row level and at the employee ID level to ensure that each employee is represented consistently in the cleaned dataset.

In [122]:
# Check for fully duplicated rows

full_row_duplicates = df_clean.duplicated().sum()
print("Number of fully duplicated rows:", full_row_duplicates)

if full_row_duplicates > 0:
    display(df_clean[df_clean.duplicated(keep=False)])
else:
    print("No fully duplicated rows found.")

Number of fully duplicated rows: 0
No fully duplicated rows found.


In [123]:
# Check for duplicated employee IDs

employee_id_duplicates = df_clean["employee_id"].duplicated().sum()
print("Number of duplicated employee_id values:", employee_id_duplicates)

if employee_id_duplicates > 0:
    dup_employee_rows = df_clean[df_clean["employee_id"].duplicated(keep=False)].sort_values("employee_id")
    display(dup_employee_rows)
else:
    print("No duplicated employee_id values found.")

Number of duplicated employee_id values: 0
No duplicated employee_id values found.


In [124]:
# Drop fully duplicated rows only if they exist

if full_row_duplicates > 0:
    df_clean = df_clean.drop_duplicates()
    print("Shape after dropping full-row duplicates:", df_clean.shape)
else:
    print("No full-row duplicates to remove.")

No full-row duplicates to remove.


No full-row duplicates and no duplicated employee IDs were found, so no rows needed to be removed at this step.

### 4.3 Data Type Inspection

We review the data types of all variables to ensure that numeric variables are stored as numeric values and categorical variables are stored as text/object values.

In [125]:
# Display the data types of all columns

dtype_report = pd.DataFrame({
    "column": df_clean.columns,
    "dtype": df_clean.dtypes.values
})

display(dtype_report)

,column,dtype
0,age,int64
1,attrition,object
2,business_travel,object
3,department,object
4,distance_from_home,int64
5,education,int64
6,education_field,object
7,employee_count,int64
8,employee_id,int64
9,gender,object


In [126]:
# Separate numeric and object columns

numeric_cols = df_clean.select_dtypes(include="number").columns.tolist()
object_cols = df_clean.select_dtypes(include="object").columns.tolist()

print("Numeric columns:")
print(numeric_cols)

print("\nObject columns:")
print(object_cols)

Numeric columns:
['age', 'distance_from_home', 'education', 'employee_count', 'employee_id', 'job_level', 'monthly_income', 'num_companies_worked', 'percent_salary_hike', 'standard_hours', 'stock_option_level', 'total_working_years', 'training_times_last_year', 'years_at_company', 'years_since_last_promotion', 'years_with_curr_manager', 'environment_satisfaction', 'job_satisfaction', 'work_life_balance', 'job_involvement', 'performance_rating']

Object columns:
['attrition', 'business_travel', 'department', 'education_field', 'gender', 'job_role', 'marital_status', 'over_18']


In [127]:
# Preview sample values in each object column

for col in object_cols:
    print("=" * 60)
    print(f"Column: {col}")
    print("Sample values:", df_clean[col].dropna().unique()[:10])

Column: attrition
Sample values: ['No' 'Yes']
Column: business_travel
Sample values: ['Travel_Rarely' 'Travel_Frequently' 'Non-Travel']
Column: department
Sample values: ['Sales' 'Research & Development' 'Human Resources']
Column: education_field
Sample values: ['Life Sciences' 'Other' 'Medical' 'Marketing' 'Technical Degree'
 'Human Resources']
Column: gender
Sample values: ['Female' 'Male']
Column: job_role
Sample values: ['Healthcare Representative' 'Research Scientist' 'Sales Executive'
 'Human Resources' 'Research Director' 'Laboratory Technician'
 'Manufacturing Director' 'Sales Representative' 'Manager']
Column: marital_status
Sample values: ['Married' 'Single' 'Divorced']
Column: over_18
Sample values: ['Y']


In [128]:
# Check for actual date/time columns in the merged dataset

date_like_cols = [
    col for col in df_clean.columns
    if col in ["date", "datetime", "timestamp", "checkin_time", "checkout_time"]
    or col.endswith("_date")
    or col.endswith("_time")
]

print("Potential date/time columns:", date_like_cols)

Potential date/time columns: []


The data types are already appropriate for the current merged dataset.  
Numeric variables are stored as numeric types, categorical variables are stored as object/text, and no actual date/time columns are present at this stage.

### 4.4 Categorical Value Cleaning

We inspect categorical columns to ensure that their values are consistent, free of unnecessary spaces, and free of unusual or duplicate category labels caused by inconsistent formatting.

In [129]:
# Identify categorical columns

categorical_cols = df_clean.select_dtypes(include="object").columns.tolist()

print("Categorical columns:")
print(categorical_cols)

Categorical columns:
['attrition', 'business_travel', 'department', 'education_field', 'gender', 'job_role', 'marital_status', 'over_18']


In [130]:
# Preview unique values before cleaning

for col in categorical_cols:
    print("=" * 60)
    print(f"Column: {col}")
    print(df_clean[col].dropna().unique()[:20])

Column: attrition
['No' 'Yes']
Column: business_travel
['Travel_Rarely' 'Travel_Frequently' 'Non-Travel']
Column: department
['Sales' 'Research & Development' 'Human Resources']
Column: education_field
['Life Sciences' 'Other' 'Medical' 'Marketing' 'Technical Degree'
 'Human Resources']
Column: gender
['Female' 'Male']
Column: job_role
['Healthcare Representative' 'Research Scientist' 'Sales Executive'
 'Human Resources' 'Research Director' 'Laboratory Technician'
 'Manufacturing Director' 'Sales Representative' 'Manager']
Column: marital_status
['Married' 'Single' 'Divorced']
Column: over_18
['Y']


In [131]:
# Remove leading and trailing spaces from categorical columns

for col in categorical_cols:
    df_clean[col] = df_clean[col].astype(str).str.strip()

In [132]:
# Re-check unique values after basic cleaning

for col in categorical_cols:
    print("=" * 60)
    print(f"Column: {col}")
    print(df_clean[col].dropna().unique()[:20])

Column: attrition
['No' 'Yes']
Column: business_travel
['Travel_Rarely' 'Travel_Frequently' 'Non-Travel']
Column: department
['Sales' 'Research & Development' 'Human Resources']
Column: education_field
['Life Sciences' 'Other' 'Medical' 'Marketing' 'Technical Degree'
 'Human Resources']
Column: gender
['Female' 'Male']
Column: job_role
['Healthcare Representative' 'Research Scientist' 'Sales Executive'
 'Human Resources' 'Research Director' 'Laboratory Technician'
 'Manufacturing Director' 'Sales Representative' 'Manager']
Column: marital_status
['Married' 'Single' 'Divorced']
Column: over_18
['Y']


In [133]:
# Show value counts for the main categorical columns

for col in ["attrition", "business_travel", "department", "education_field", "gender", "job_role", "marital_status", "over_18"]:
    if col in df_clean.columns:
        print("=" * 60)
        print(f"Value counts for: {col}")
        print(df_clean[col].value_counts(dropna=False))
        print()

Value counts for: attrition
attrition
No     3699
Yes     711
Name: count, dtype: int64

Value counts for: business_travel
business_travel
Travel_Rarely        3129
Travel_Frequently     831
Non-Travel            450
Name: count, dtype: int64

Value counts for: department
department
Research & Development    2883
Sales                     1338
Human Resources            189
Name: count, dtype: int64

Value counts for: education_field
education_field
Life Sciences       1818
Medical             1392
Marketing            477
Technical Degree     396
Other                246
Human Resources       81
Name: count, dtype: int64

Value counts for: gender
gender
Male      2646
Female    1764
Name: count, dtype: int64

Value counts for: job_role
job_role
Sales Executive              978
Research Scientist           876
Laboratory Technician        777
Manufacturing Director       435
Healthcare Representative    393
Manager                      306
Sales Representative         249
Research Dire

The categorical values are already consistent across the dataset.  
No major issues such as extra spaces, mixed capitalization, or unusual category labels were found.  
Only basic string cleaning was applied as a precaution.

### 4.5 Removing Uninformative Columns

Some columns do not provide useful variation for clustering.  
In particular, constant columns with only one unique value do not help separate employee groups and should be removed.

In [134]:
# Check columns with only one unique value

constant_cols = [col for col in df_clean.columns if df_clean[col].nunique(dropna=False) == 1]

print("Constant columns:")
print(constant_cols)

Constant columns:
['employee_count', 'over_18', 'standard_hours']


In [135]:
# Show the number of unique values in each column

unique_counts = pd.DataFrame({
    "column": df_clean.columns,
    "n_unique": [df_clean[col].nunique(dropna=False) for col in df_clean.columns]
}).sort_values(by="n_unique")

display(unique_counts.head(10))

,column,n_unique
15,over_18,1
17,standard_hours,1
7,employee_count,1
28,performance_rating,2
9,gender,2
1,attrition,2
3,department,3
2,business_travel,3
12,marital_status,3
25,job_satisfaction,4


In [136]:
# Define ID columns and columns to exclude from clustering later

id_cols = ["employee_id"]

print("ID columns:", id_cols)
print("Constant columns:", constant_cols)

ID columns: ['employee_id']
Constant columns: ['employee_count', 'over_18', 'standard_hours']


In [137]:
# Drop constant columns but keep employee_id for tracking

df_clean = df_clean.drop(columns=constant_cols)

print("Shape after dropping constant columns:", df_clean.shape)
display(df_clean.head())

Shape after dropping constant columns: (4410, 26)


,age,attrition,business_travel,department,distance_from_home,education,education_field,employee_id,gender,job_level,...,total_working_years,training_times_last_year,years_at_company,years_since_last_promotion,years_with_curr_manager,environment_satisfaction,job_satisfaction,work_life_balance,job_involvement,performance_rating
0,51,No,Travel_Rarely,Sales,6,2,Life Sciences,1,Female,1,...,1.0,6,1,0,0,3.0,4.0,2.0,3,3
1,31,Yes,Travel_Frequently,Research & Development,10,1,Life Sciences,2,Female,1,...,6.0,3,5,1,4,3.0,2.0,4.0,2,4
2,32,No,Travel_Frequently,Research & Development,17,4,Other,3,Male,4,...,5.0,2,5,0,3,2.0,2.0,1.0,3,3
3,38,No,Non-Travel,Research & Development,2,5,Life Sciences,4,Male,3,...,13.0,5,8,7,5,4.0,4.0,3.0,2,3
4,32,No,Travel_Rarely,Research & Development,10,1,Medical,5,Male,1,...,9.0,2,6,0,4,4.0,1.0,3.0,3,3


In [138]:
# Keep track of columns to exclude from clustering input later

exclude_from_clustering = ["employee_id"] + constant_cols

print("Columns to exclude from clustering input later:")
print(exclude_from_clustering)

Columns to exclude from clustering input later:
['employee_id', 'employee_count', 'over_18', 'standard_hours']


The constant columns `employee_count`, `over_18`, and `standard_hours` were removed because they do not help distinguish employee groups.  
The column `employee_id` is kept for tracking purposes, but it will be excluded from clustering input later.

### 4.6 Preliminary Outlier Check

A preliminary outlier screening is conducted for numeric variables because clustering methods such as K-Means can be sensitive to extreme values.

In [139]:
# Select numeric columns for outlier screening

numeric_for_outlier = df_clean.select_dtypes(include="number").columns.tolist()

if "employee_id" in numeric_for_outlier:
    numeric_for_outlier.remove("employee_id")

print("Numeric columns for outlier check:")
print(numeric_for_outlier)

Numeric columns for outlier check:
['age', 'distance_from_home', 'education', 'job_level', 'monthly_income', 'num_companies_worked', 'percent_salary_hike', 'stock_option_level', 'total_working_years', 'training_times_last_year', 'years_at_company', 'years_since_last_promotion', 'years_with_curr_manager', 'environment_satisfaction', 'job_satisfaction', 'work_life_balance', 'job_involvement', 'performance_rating']


In [140]:
# Detect outliers using the IQR rule

outlier_summary = []

for col in numeric_for_outlier:
    q1 = df_clean[col].quantile(0.25)
    q3 = df_clean[col].quantile(0.75)
    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    outlier_count = ((df_clean[col] < lower_bound) | (df_clean[col] > upper_bound)).sum()
    outlier_pct = round(outlier_count / len(df_clean) * 100, 2)

    outlier_summary.append({
        "column": col,
        "q1": q1,
        "q3": q3,
        "iqr": iqr,
        "lower_bound": lower_bound,
        "upper_bound": upper_bound,
        "outlier_count": outlier_count,
        "outlier_pct": outlier_pct
    })

outlier_report = pd.DataFrame(outlier_summary).sort_values(
    by="outlier_count", ascending=False
).reset_index(drop=True)

display(outlier_report)

,column,q1,q3,iqr,lower_bound,upper_bound,outlier_count,outlier_pct
0,training_times_last_year,2.0,3.0,1.0,0.5,4.5,714,16.19
1,performance_rating,3.0,3.0,0.0,3.0,3.0,678,15.37
2,monthly_income,29110.0,83800.0,54690.0,-52925.0,165835.0,342,7.76
3,years_since_last_promotion,0.0,3.0,3.0,-4.5,7.5,321,7.28
4,years_at_company,3.0,9.0,6.0,-6.0,18.0,312,7.07
5,stock_option_level,0.0,1.0,1.0,-1.5,2.5,255,5.78
6,total_working_years,6.0,15.0,9.0,-7.5,28.5,189,4.29
7,num_companies_worked,1.0,4.0,3.0,-3.5,8.5,156,3.54
8,years_with_curr_manager,2.0,7.0,5.0,-5.5,14.5,42,0.95
9,education,2.0,4.0,2.0,-1.0,7.0,0,0.00


In [141]:
# Show only columns with detected outliers

outlier_only = outlier_report[outlier_report["outlier_count"] > 0]
display(outlier_only)

,column,q1,q3,iqr,lower_bound,upper_bound,outlier_count,outlier_pct
0,training_times_last_year,2.0,3.0,1.0,0.5,4.5,714,16.19
1,performance_rating,3.0,3.0,0.0,3.0,3.0,678,15.37
2,monthly_income,29110.0,83800.0,54690.0,-52925.0,165835.0,342,7.76
3,years_since_last_promotion,0.0,3.0,3.0,-4.5,7.5,321,7.28
4,years_at_company,3.0,9.0,6.0,-6.0,18.0,312,7.07
5,stock_option_level,0.0,1.0,1.0,-1.5,2.5,255,5.78
6,total_working_years,6.0,15.0,9.0,-7.5,28.5,189,4.29
7,num_companies_worked,1.0,4.0,3.0,-3.5,8.5,156,3.54
8,years_with_curr_manager,2.0,7.0,5.0,-5.5,14.5,42,0.95


In [142]:
# Preview extreme values for the top columns with detected outliers

top_outlier_cols = outlier_only["column"].head(4).tolist()

for col in top_outlier_cols:
    print("=" * 60)
    print(f"Column: {col}")
    print("Smallest values:")
    print(df_clean[col].sort_values().head(10).tolist())
    print("Largest values:")
    print(df_clean[col].sort_values(ascending=False).head(10).tolist())

Column: training_times_last_year
Smallest values:
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Largest values:
[6, 6, 6, 6, 6, 6, 6, 6, 6, 6]
Column: performance_rating
Smallest values:
[3, 3, 3, 3, 3, 3, 3, 3, 3, 3]
Largest values:
[4, 4, 4, 4, 4, 4, 4, 4, 4, 4]
Column: monthly_income
Smallest values:
[10090, 10090, 10090, 10510, 10510, 10510, 10520, 10520, 10520, 10810]
Largest values:
[199990, 199990, 199990, 199730, 199730, 199730, 199430, 199430, 199430, 199260]
Column: years_since_last_promotion
Smallest values:
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Largest values:
[15, 15, 15, 15, 15, 15, 15, 15, 15, 15]


At this stage, outliers are only identified and documented.  
They are not removed immediately, because some extreme values may still be valid employee records.  
These findings will be considered later when preparing the clustering input, especially for K-Means.

### 4.7 Saving the Cleaned Dataset

The cleaned dataset is saved for the next stages of the project, including EDA, feature engineering, and clustering.

In [143]:
# Save the cleaned dataset

df_clean.to_csv(PROCESSED_DIR / "cleaned_data.csv", index=False)

print("Saved cleaned_data.csv successfully.")
print("File path:", PROCESSED_DIR / "cleaned_data.csv")
print("Final cleaned shape:", df_clean.shape)

Saved cleaned_data.csv successfully.
File path: ..\data\processed\cleaned_data.csv
Final cleaned shape: (4410, 26)


In [144]:
# Reload the saved file to verify it

df_check = pd.read_csv(PROCESSED_DIR / "cleaned_data.csv")

print("Reloaded cleaned_data shape:", df_check.shape)
display(df_check.head())

Reloaded cleaned_data shape: (4410, 26)


,age,attrition,business_travel,department,distance_from_home,education,education_field,employee_id,gender,job_level,...,total_working_years,training_times_last_year,years_at_company,years_since_last_promotion,years_with_curr_manager,environment_satisfaction,job_satisfaction,work_life_balance,job_involvement,performance_rating
0,51,No,Travel_Rarely,Sales,6,2,Life Sciences,1,Female,1,...,1.0,6,1,0,0,3.0,4.0,2.0,3,3
1,31,Yes,Travel_Frequently,Research & Development,10,1,Life Sciences,2,Female,1,...,6.0,3,5,1,4,3.0,2.0,4.0,2,4
2,32,No,Travel_Frequently,Research & Development,17,4,Other,3,Male,4,...,5.0,2,5,0,3,2.0,2.0,1.0,3,3
3,38,No,Non-Travel,Research & Development,2,5,Life Sciences,4,Male,3,...,13.0,5,8,7,5,4.0,4.0,3.0,2,3
4,32,No,Travel_Rarely,Research & Development,10,1,Medical,5,Male,1,...,9.0,2,6,0,4,4.0,1.0,3.0,3,3


### 4.8 Cleaning Summary

A short summary is recorded to document the main cleaning decisions and support the final report writing.

In [145]:
# Create a cleaning summary text
cleaning_summary = f"""# Cleaning Summary

## Dataset shapes
- Before cleaning: {merged_df.shape}
- After cleaning: {df_clean.shape}

## Columns dropped
- {", ".join(constant_cols)}

## Missing values
- Missing values were found in:
  - work_life_balance
  - environment_satisfaction
  - job_satisfaction
  - num_companies_worked
  - total_working_years
- All missing percentages were low, so median imputation was applied to numeric columns.

## Duplicate check
- Fully duplicated rows: {full_row_duplicates}
- Duplicated employee IDs: {employee_id_duplicates}
- No duplicate rows were removed because no true duplicates were found.

## Data type review
- Numeric variables already had appropriate numeric types.
- Categorical variables were stored as object/text.
- No actual date/time columns were present in the merged dataset.

## Categorical values
- No major categorical inconsistencies were found.
- Basic string cleaning was applied.

## Clustering note
- The following columns will be excluded from clustering input later:
  - {", ".join(exclude_from_clustering)}
"""

print(cleaning_summary)

# Cleaning Summary

## Dataset shapes
- Before cleaning: (4410, 29)
- After cleaning: (4410, 26)

## Columns dropped
- employee_count, over_18, standard_hours

## Missing values
- Missing values were found in:
  - work_life_balance
  - environment_satisfaction
  - job_satisfaction
  - num_companies_worked
  - total_working_years
- All missing percentages were low, so median imputation was applied to numeric columns.

## Duplicate check
- Fully duplicated rows: 0
- Duplicated employee IDs: 0
- No duplicate rows were removed because no true duplicates were found.

## Data type review
- Numeric variables already had appropriate numeric types.
- Categorical variables were stored as object/text.
- No actual date/time columns were present in the merged dataset.

## Categorical values
- No major categorical inconsistencies were found.
- Basic string cleaning was applied.

## Clustering note
- The following columns will be excluded from clustering input later:
  - employee_id, employee_count, 

In [146]:
# Save the cleaning summary as a markdown file

with open(PROCESSED_DIR / "cleaning_summary.md", "w", encoding="utf-8") as f:
    f.write(cleaning_summary)

print("Saved cleaning_summary.md successfully.")

Saved cleaning_summary.md successfully.


## 5. EDA

## 6. Feature Engineering / Preprocessing

## 7. Clustering Models

## 8. Evaluation

## 9. Conclusion